In [ ]:
import os
import json
import datetime
import re
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor
from typing import Dict, List, Any, Tuple, Set
from collections import defaultdict
from dotenv import load_dotenv
from datasets import load_dataset
from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic
from langchain_deepseek import ChatDeepSeek
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_mistralai import ChatMistralAI
from langchain_xai import ChatXAI
from langchain_core.messages import HumanMessage
from langchain_core.language_models import BaseChatModel
import numpy as np
from tqdm import tqdm
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
from module.utils import extract_skeleton

def load_environment():
    """Load environment variables"""
    load_dotenv()

def load_ground_truth() -> Dict[str, str]:
    """Load ground truth buggy file paths."""
    with open("./ground_truth/bug_paths.json", "r") as f:
        return json.load(f)

def load_candidates(candidates_path: str) -> Dict[str, List[str]]:
    """Load candidate file paths from JSON."""
    with open(candidates_path, "r") as f:
        return json.load(f)

def load_model_history(provider: str) -> Dict[str, Dict[str, List[str]]]:
    """Load history for a specific model provider."""
    history_path = f"./localization_ranking_results/model_history/{provider}/ranking_history.json"
    
    if not os.path.exists(history_path):
        # Create the directory if it doesn't exist
        os.makedirs(os.path.dirname(history_path), exist_ok=True)
        return {}
    
    try:
        with open(history_path, "r") as f:
            return json.load(f)
    except (json.JSONDecodeError, FileNotFoundError):
        return {}

def create_llm(llm_name: str, model_name: str, temperature: float, max_tokens: int) -> BaseChatModel:
    """Create language model instance based on configuration"""
    if llm_name == "chatgpt":
        return ChatOpenAI(
            model_name=model_name,
            openai_api_key=os.getenv("OPENAI_API_KEY"),
            temperature=temperature,
            max_tokens=max_tokens
        )
    elif llm_name == "claude":
        return ChatAnthropic(
            model_name=model_name,
            anthropic_api_key=os.getenv("ANTHROPIC_API_KEY"),
            temperature=temperature,
            max_tokens=max_tokens
        )
    elif llm_name == "deepseek":
        return ChatDeepSeek(
            model=model_name,
            api_key=os.getenv("DEEPSEEK_API_KEY"),
            temperature=temperature,
            max_tokens=max_tokens
        )
    elif llm_name == "gemini":
        return ChatGoogleGenerativeAI(
            model=model_name,
            google_api_key=os.getenv("GOOGLE_API_KEY"),
            temperature=temperature,
            max_tokens=max_tokens
        )
    elif llm_name == "mistral":
        return ChatMistralAI(
            model=model_name,
            mistral_api_key=os.getenv("MISTRAL_API_KEY"),
            temperature=temperature,
            max_tokens=max_tokens
        )
    elif llm_name == "grok":
        return ChatXAI(
            model=model_name,
            api_key=os.getenv("XAI_API_KEY"),
            temperature=temperature,
            max_tokens=max_tokens
        )
    else:
        raise ValueError(f"Unsupported LLM: {llm_name}")

In [ ]:
def build_ranking_prompt(problem_statement: str, file_skeletons: Dict[str, str], file_paths: List[str]) -> str:
    """Build the prompt for ranking file paths by likelihood of containing bugs."""
    
    # Format the file skeletons and paths for the prompt
    files_section = ""
    for i, path in enumerate(file_paths, 1):
        skeleton = file_skeletons.get(path, "Unable to extract skeleton")
        files_section += f"""
<file index="{i}">
  <path>{path}</path>
  <skeleton>
{skeleton}
  </skeleton>
</file>

"""
    
    prompt = f"""
You are an AI assistant tasked with analyzing and ranking files based on their likelihood of containing a bug related to a given issue description. You will be provided with an issue description and a list of candidate files. Your goal is to rank these files from highest to lowest probability of containing the bug related to the described issue.

First, carefully read and understand the following issue description:

<issue_description>
{problem_statement}
</issue_description>

Now, you will be presented with a list of candidate files. Each file entry will contain the file's structure, declarations, function signatures, and class definitions, but omits implementation details.

Here are the candidate files:
<candidate_files>
{files_section}
</candidate_files>


IMPORTANT: Your final output must ONLY contain a numbered list of file paths, ranked from highest to lowest probability of containing the bug. Do not include any justifications, explanations, or additional analysis in the final output.

Example of the required output format:

<ranking>
1. /path/to/file1.py
2. /path/to/file2.py
3. /path/to/file3.py
...
</ranking>

"""
    return prompt


In [ ]:
def extract_ranking(model_response: str, file_paths: List[str]) -> List[str]:
    """
    Extract file rankings from the model's response.
    
    Args:
        model_response: The response from the model containing the ranking.
        file_paths: List of file paths that should be present in the ranking.
    
    Returns:
        A list of file paths in ranked order (highest to lowest probability).
    """
    # Extract content between <ranking> and </ranking> tags
    ranking_pattern = re.compile(r'<ranking>(.*?)</ranking>', re.DOTALL)
    ranking_match = ranking_pattern.search(model_response)
    
    if not ranking_match:
        print("WARNING: No <ranking> tags found in the model response.")
        return []
    
    ranking_content = ranking_match.group(1).strip()
    
    # Extract file paths from the numbered list
    ranked_paths = []
    for line in ranking_content.split('\n'):
        line = line.strip()
        if not line:
            continue
            
        # Match numbered list pattern (e.g., "1. /path/to/file.py")
        match = re.match(r'^\d+\.\s+(.+?)$', line)
        if match:
            path = match.group(1).strip()
            ranked_paths.append(path)
    
    # Check if all extracted paths exist in the provided file_paths
    filtered_ranked_paths = []
    for path in ranked_paths:
        if path not in file_paths:
            print(f"WARNING: Extracted path '{path}' not found in the provided file paths.")
        else:
            filtered_ranked_paths.append(path)
    ranked_paths = filtered_ranked_paths
    
    # Check if all file paths are included in the ranking
    ranked_set = set(ranked_paths)
    file_set = set(file_paths)
    
    if len(ranked_set) != len(file_paths):
        print(f"WARNING: Expected {len(file_paths)} files in ranking, but got {len(ranked_set)}.")
        
        missing_paths = file_set - ranked_set
        if missing_paths:
            print(f"WARNING: The following paths are missing from the ranking: {missing_paths}")
        
        extra_paths = ranked_set - file_set
        if extra_paths:
            print(f"WARNING: The following paths in the ranking were not in the provided file paths: {extra_paths}")
    
    return ranked_paths


@retry(
    stop=stop_after_attempt(5),
    wait=wait_exponential(multiplier=1, min=120, max=1800),
    retry=retry_if_exception_type((Exception))
)
def get_llm_ranking(llm, prompt, llm_name, instance_id, paths, history_cache):
    """Get ranking from LLM with caching and fallback retry to Claude 3.5 Sonnet on error."""
    # Create a stable key for this set of paths
    paths_key = "|".join(sorted(paths))
    
    # Check if result is already in cache
    if instance_id in history_cache and paths_key == history_cache[instance_id]:
        print(f"Using cached ranking for {instance_id}")
        return history_cache[instance_id][paths_key], history_cache[instance_id][paths_key], llm_name
    
    try:
        response = llm.invoke([HumanMessage(content=prompt)])
        raw_response = response.content
        
        # Process response to extract ranking
        # ranked_paths, is_valid_format = extract_ranking(raw_response, paths)
        ranked_paths = extract_ranking(raw_response, paths)
        
        # if not is_valid_format:
        #     print(f"WARNING: Response not in expected format for {llm_name}: {instance_id}")
        #     print(f"Response: {raw_response[:500]}...")
            
        #     # If extraction failed but we got a response, try a simple fallback
        #     if not ranked_paths:
        #         # Just return the original paths (unranked) to avoid complete failure
        #         ranked_paths = paths
        
        # Return raw response and extracted ranking
        return raw_response, ranked_paths, llm_name

    except Exception as e:
        error_msg = str(e)
        # If it's a rate limit (429) or server error (500, 502), re-raise to let the retry decorator handle it
        if any(code in error_msg for code in ["429", "500", "502"]):
            print(f"Rate limit or server error ({error_msg}) for {llm_name}. Retrying...")
            raise e
            
        # For other errors, fallback to Claude
        print(f"Error during LLM call ({llm.__class__.__name__} - {getattr(llm, 'model_name', 'unknown')}): {e}")
        print("Falling back to Claude 3.5 Sonnet...")

        try:
            # Load Claude history cache
            claude_history = load_model_history("claude")
            
            # Check if the result is already in Claude's cache
            if instance_id in claude_history and paths_key == claude_history[instance_id]:
                print(f"Using cached Claude ranking for {instance_id}")
                return "Using cached Claude result", claude_history[instance_id][paths_key], "claude"
            
            # Otherwise, call Claude
            fallback_llm = ChatAnthropic(
                model_name="claude-3-5-sonnet-20241022",
                anthropic_api_key=os.getenv("ANTHROPIC_API_KEY"),
                temperature=0.0,
                max_tokens=8192
            )
            response = fallback_llm.invoke([HumanMessage(content=prompt)])
            raw_response = response.content
            
            # Process response
            ranked_paths, is_valid_format = extract_ranking(raw_response, paths)
            
            if not is_valid_format:
                print(f"WARNING: Fallback response not in expected format: {instance_id}")
                print(f"Response: {raw_response[:500]}...")
                
                # If extraction failed but we got a response, use a simple fallback
                if not ranked_paths:
                    ranked_paths = paths
            
            # Return with provider info
            return raw_response, ranked_paths, "claude"
            
        except Exception as fallback_error:
            print(f"Fallback Claude 3.5 Sonnet also failed: {fallback_error}")
            # Last resort: return original paths
            return f"Error: {str(fallback_error)}", paths, "error"

def process_single_instance(task_data: Tuple) -> Dict[str, Any]:
    """Process a single instance's ranking task."""
    instance_id, file_paths, problem_statement, llm_config, raw_outputs_dir, history_cache = task_data
    
    llm_name = llm_config["llm"]
    model_name = llm_config["model_name"]
    
    try:
        # Skip if only one file - no ranking needed
        if len(file_paths) <= 1:
            return {
                "instance_id": instance_id,
                "paths": file_paths,
                "ranked_paths": file_paths,  # Same as input if only one path
                "llm": llm_name,
                "model_name": model_name,
                "raw_response": "Only one file, no ranking needed",
                "provider": llm_name
            }
        
        # Get file skeletons
        file_skeletons = {}
        for path in file_paths:
            file_path = os.path.join("./codebases", instance_id, path)
            
            # Check if file exists
            if not os.path.exists(file_path):
                print(f"Warning: File does not exist: {file_path}")
                file_skeletons[path] = "File does not exist"
                continue
            
            # Read file content and extract skeleton
            try:
                with open(file_path, "r", encoding="utf-8", errors="replace") as f:
                    file_content = f.read()
                file_skeletons[path] = extract_skeleton(file_content)
            except Exception as e:
                print(f"Error extracting skeleton for {file_path}: {e}")
                file_skeletons[path] = f"Error extracting skeleton: {str(e)}"
        
        # Build the prompt
        prompt = build_ranking_prompt(problem_statement, file_skeletons, file_paths)
        
        # Create a new LLM instance for this process
        llm = create_llm(llm_name, model_name, llm_config["temperature"], llm_config["max_tokens"])
        
        # Get LLM ranking with retry decorator and caching
        raw_response, ranked_paths, provider = get_llm_ranking(llm, prompt, llm_name, instance_id, file_paths, history_cache)
        
        # Save raw response for this instance
        instance_dir = os.path.join(raw_outputs_dir, instance_id)
        os.makedirs(instance_dir, exist_ok=True)
        
        raw_output_path = os.path.join(instance_dir, f"ranking_{llm_name}_{model_name}.json")
        with open(raw_output_path, "w", encoding="utf-8") as f:
            json.dump({
                "instance_id": instance_id,
                "paths": file_paths,
                "ranked_paths": ranked_paths,
                "llm": llm_name,
                "model_name": model_name,
                "raw_response": raw_response
            }, f, indent=2, ensure_ascii=False)
        
        return {
            "instance_id": instance_id,
            "paths": file_paths,
            "ranked_paths": ranked_paths,
            "llm": llm_name,
            "model_name": model_name,
            "raw_response": raw_response,
            "provider": provider
        }
        
    except Exception as e:
        print(f"Error processing {instance_id} with {llm_name}/{model_name}: {e}")
        return {
            "instance_id": instance_id,
            "paths": file_paths,
            "ranked_paths": file_paths,  # Return original paths on error
            "llm": llm_name,
            "model_name": model_name,
            "raw_response": f"Error: {str(e)}",
            "provider": llm_name
        }

def save_model_history(provider: str, new_history_entries: Dict[str, Dict[str, List[str]]]):
    """Save ranking history for a specific model provider, merging with existing data."""
    history_path = f"./localization_ranking_results/model_history/{provider}/ranking_history.json"
    
    # Create the directory if it doesn't exist
    os.makedirs(os.path.dirname(history_path), exist_ok=True)
    
    # Load existing history if present
    existing_history = {}
    if os.path.exists(history_path):
        try:
            with open(history_path, "r") as f:
                existing_history = json.load(f)
        except (json.JSONDecodeError, FileNotFoundError):
            existing_history = {}
    
    # Merge new entries with existing history
    updated_entries = 0
    for instance_id, path_entries in new_history_entries.items():
        if instance_id not in existing_history:
            existing_history[instance_id] = {}
        
        for paths_key, ranked_paths in path_entries.items():
            # Only count as an update if it's a new entry or the ranking changed
            if paths_key not in existing_history[instance_id] or existing_history[instance_id][paths_key] != ranked_paths:
                updated_entries += 1
            
            existing_history[instance_id][paths_key] = ranked_paths
    
    # Save the merged history
    with open(history_path, "w") as f:
        json.dump(existing_history, f, indent=2)
    
    if updated_entries > 0:
        print(f"Updated history for {provider} with {updated_entries} new entries")
    else:
        print(f"No new entries to update for {provider}")
    
    return existing_history

def calculate_accuracy_at_k(results: List[Dict[str, Any]], ground_truth: Dict[str, str], k: int) -> float:
    """Calculate accuracy@k for ranking results."""
    correct_at_k = 0
    total = 0
    
    for result in results:
        instance_id = result["instance_id"]
        ranked_paths = result["ranked_paths"]
        
        # Skip if no ground truth or no ranking
        if instance_id not in ground_truth or not ranked_paths:
            continue
        
        gt_path = ground_truth[instance_id]
        
        # Check if ground truth path is in top-k
        if gt_path in ranked_paths[:k]:
            correct_at_k += 1
        
        total += 1
    
    return correct_at_k / total if total > 0 else 0

def run_ranking(ranking_config):
    """Run the ranking process for candidate buggy files."""
    # Load environment variables
    load_environment()
    
    # Load ground truth
    print("Loading ground truth data...")
    ground_truth = load_ground_truth()
    
    # Load candidate paths
    print(f"Loading candidate file paths from {ranking_config['candidates_path']}")
    candidates = load_candidates(ranking_config["candidates_path"])
    
    # Filter instances with multiple paths (to be ranked)
    multi_path_instances = {instance_id: paths for instance_id, paths in candidates.items() if len(paths) > 1}
    single_path_instances = {instance_id: paths for instance_id, paths in candidates.items() if len(paths) == 1}
    
    print(f"Total instances: {len(candidates)}")
    print(f"Instances with single path (no ranking needed): {len(single_path_instances)}")
    print(f"Instances with multiple paths (to be ranked): {len(multi_path_instances)}")
    
    # Load dataset for problem statements
    print("Loading SWE-bench dataset...")
    dataset = load_dataset("princeton-nlp/SWE-bench_Lite", split="test")
    dataset_dict = {item["instance_id"]: item for item in dataset}
    
    # Create output directory
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    output_dir = Path(f"./localization_ranking_results/{timestamp}")
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Create raw outputs directory
    raw_outputs_dir = str(output_dir / "raw_outputs")
    os.makedirs(raw_outputs_dir, exist_ok=True)
    
    # Save config
    with open(output_dir / "config.json", "w") as f:
        save_config = ranking_config.copy()
        # Add additional information to config
        save_config["timestamp"] = timestamp
        save_config["num_instances"] = len(candidates)
        save_config["num_multi_path_instances"] = len(multi_path_instances)
        save_config["num_single_path_instances"] = len(single_path_instances)
        json.dump(save_config, f, indent=2)
    
    # Load model history cache
    print(f"Loading history cache for {ranking_config['llm']}...")
    history_cache = load_model_history(ranking_config['llm'])
    
    # Prepare tasks - one task per instance
    all_tasks = []
    
    llm_name = ranking_config["llm"]
    model_name = ranking_config["model_name"]
    
    print(f"Preparing tasks for {llm_name}/{model_name}...")
    
    # Add tasks for multi-path instances
    for instance_id, paths in multi_path_instances.items():
        # Skip if instance is not in dataset
        if instance_id not in dataset_dict:
            print(f"Warning: Instance {instance_id} not found in dataset")
            continue
        
        problem_statement = dataset_dict[instance_id]["problem_statement"]
        
        # Create a task for each instance
        all_tasks.append((
            instance_id, 
            paths, 
            problem_statement, 
            ranking_config,
            raw_outputs_dir,
            history_cache
        ))
    
    # Process tasks with multiprocessing
    print(f"Processing {len(all_tasks)} ranking tasks using {ranking_config['num_processes']} processes...")
    
    if ranking_config["num_processes"] > 1:
        multi_path_results = []
        with ProcessPoolExecutor(max_workers=ranking_config["num_processes"]) as executor:
            # Use tqdm to show progress
            for result in tqdm(executor.map(process_single_instance, all_tasks), 
                               total=len(all_tasks), 
                               desc="Processing multi-path instances"):
                multi_path_results.append(result)
    else:
        multi_path_results = []
        for task in tqdm(all_tasks, desc="Processing multi-path instances"):
            multi_path_results.append(process_single_instance(task))
    
    # Add single-path instances to results
    single_path_results = []
    for instance_id, paths in single_path_instances.items():
        single_path_results.append({
            "instance_id": instance_id,
            "paths": paths,
            "ranked_paths": paths,
            "llm": llm_name,
            "model_name": model_name,
            "raw_response": "Single path, no ranking needed",
            "provider": llm_name
        })
    
    # Combine results
    all_results = multi_path_results + single_path_results
    
    # Save all raw results
    with open(output_dir / "raw_results.json", "w", encoding="utf-8") as f:
        json.dump(all_results, f, indent=2, ensure_ascii=False)
    
    # Process results to update history caches for each provider
    print("Updating history caches from results...")
    
    # Group results by provider
    provider_history = defaultdict(lambda: defaultdict(dict))
    for result in multi_path_results:  # Only update history for multi-path results
        instance_id = result["instance_id"]
        paths = result["paths"]
        ranked_paths = result["ranked_paths"]
        provider = result["provider"]
        
        # Create a stable key for this set of paths
        paths_key = "|".join(sorted(paths))
        
        provider_history[provider][instance_id][paths_key] = ranked_paths
    
    # Save each provider's results to its history cache
    for provider, history_entries in provider_history.items():
        if history_entries:
            print(f"Saving results for provider: {provider}")
            save_model_history(provider, history_entries)
    
    # Calculate accuracy for different k values
    print("Calculating accuracy metrics...")
    accuracy_metrics = {}
    
    for k in range(1, 6):  # Calculate accuracy@1 through accuracy@5
        if k == 1:
            # For accuracy@1, we consider both single-path and multi-path instances
            accuracy = calculate_accuracy_at_k(all_results, ground_truth, k)
        else:
            # For accuracy@k where k > 1, we only consider multi-path instances
            # (single-path instances don't have a meaningful "top-k")
            accuracy = calculate_accuracy_at_k(multi_path_results, ground_truth, k)
        
        accuracy_metrics[f"accuracy@{k}"] = accuracy
    
    # Save accuracy metrics
    with open(output_dir / "accuracy_metrics.json", "w") as f:
        json.dump(accuracy_metrics, f, indent=2)
    
    # Print results
    print("\nAccuracy metrics:")
    for k, acc in accuracy_metrics.items():
        print(f"  {k}: {acc:.4f}")
    
    print(f"\nRanking complete. Results saved to {output_dir}")
    
    return all_results, accuracy_metrics

In [ ]:
# Configuration for ranking
ranking_config = {
    # LLM configuration - specify which model to use
    "llm": "chatgpt",  # Options: chatgpt, claude, deepseek, gemini, mistral, grok
    "model_name": "gpt-4o-2024-08-06",  # claude-3-5-sonnet-20241022 gpt-4o-2024-08-06
    "temperature": 0.0,
    "max_tokens": 8192,
    
    # Path to load candidate paths
    "candidates_path": "./localization_combination_results/union/results/best_top_5_results.json",
    # "candidates_path": "./localization_candidates/voting_top5union_gptclaudedeepseek.json",
    
    # Execution configuration
    "num_processes": 1,
}

results, metrics = run_ranking(ranking_config)
print("\nRanking completed successfully.")